In [ ]:
# Environment note: this notebook was built and run on Kaggle. Attach the dataset via
# "+ Add Data" -> search "chest-xray-pneumonia" (paultimothymooney/chest-xray-pneumonia)
# before running -- see the data-loading cell below for path details.


In [ ]:
import copy
import glob
import os
import random
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
import torchvision
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


# Set seed for reproducibility across PyTorch and NumPy
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    # NOTE: this makes the *split* and initial weights reproducible, but does not
    # guarantee bit-identical training runs on GPU -- some CUDA ops (and DataLoader
    # worker shuffling with num_workers>0) remain nondeterministic unless you also set
    # torch.use_deterministic_algorithms(True) and pass a seeded `generator=` to the
    # DataLoader / a `worker_init_fn`. See the README's Reproducibility Note: even with
    # this seeding, repeated runs of this notebook showed real variance in the trained
    # models' calibration metrics -- that variance is reported explicitly, not hidden.


seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")


In [ ]:
class NumPyConv2D:
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        
        # Initialize weights and biases
        self.W = np.random.randn(out_channels, in_channels, kernel_size, kernel_size).astype(np.float32) * 0.1
        self.b = np.zeros(out_channels, dtype=np.float32)
        
    def forward(self, x):
        self.x = x
        N, C, H, W = x.shape
        K = self.kernel_size
        P = self.padding
        S = self.stride
        
        if P > 0:
            self.x_padded = np.pad(x, ((0, 0), (0, 0), (P, P), (P, P)), mode='constant')
        else:
            self.x_padded = x
            
        H_out = (H + 2 * P - K) // S + 1
        W_out = (W + 2 * P - K) // S + 1
        
        out = np.zeros((N, self.out_channels, H_out, W_out), dtype=np.float32)
        
        for n in range(N):
            for c_out in range(self.out_channels):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start, w_start = i * S, j * S
                        patch = self.x_padded[n, :, h_start:h_start+K, w_start:w_start+K]
                        out[n, c_out, i, j] = np.sum(patch * self.W[c_out]) + self.b[c_out]
        return out

    def backward(self, dout):
        N, C, H, W = self.x.shape
        K = self.kernel_size
        P = self.padding
        S = self.stride
        _, _, H_out, W_out = dout.shape
        
        dW = np.zeros_like(self.W)
        db = np.zeros_like(self.b)
        dx_padded = np.zeros_like(self.x_padded)
        
        for n in range(N):
            for c_out in range(self.out_channels):
                db[c_out] += np.sum(dout[n, c_out])
                for i in range(H_out):
                    for j in range(W_out):
                        h_start, w_start = i * S, j * S
                        patch = self.x_padded[n, :, h_start:h_start+K, w_start:w_start+K]
                        dW[c_out] += patch * dout[n, c_out, i, j]
                        dx_padded[n, :, h_start:h_start+K, w_start:w_start+K] += self.W[c_out] * dout[n, c_out, i, j]
                        
        if P > 0:
            dx = dx_padded[:, :, P:-P, P:-P]
        else:
            dx = dx_padded
            
        return dx, dW, db


class NumPyMaxPool2D:
    def __init__(self, kernel_size=2, stride=2):
        self.kernel_size = kernel_size
        self.stride = stride

    def forward(self, x):
        self.x = x
        N, C, H, W = x.shape
        K = self.kernel_size
        S = self.stride
        
        H_out = (H - K) // S + 1
        W_out = (W - K) // S + 1
        
        out = np.zeros((N, C, H_out, W_out), dtype=np.float32)
        self.arg_max = {}
        
        for n in range(N):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start, w_start = i * S, j * S
                        patch = x[n, c, h_start:h_start+K, w_start:w_start+K]
                        out[n, c, i, j] = np.max(patch)
                        idx = np.unravel_index(np.argmax(patch), patch.shape)
                        self.arg_max[(n, c, i, j)] = (h_start + idx[0], w_start + idx[1])
        return out

    def backward(self, dout):
        N, C, H, W = self.x.shape
        _, _, H_out, W_out = dout.shape
        dx = np.zeros_like(self.x)
        
        for n in range(N):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        max_h, max_w = self.arg_max[(n, c, i, j)]
                        dx[n, c, max_h, max_w] += dout[n, c, i, j]
        return dx


def test_numpy_vs_pytorch():
    np.random.seed(42)
    torch.manual_seed(42)

    # Input tensor shape: (batch_size=2, channels=3, height=16, width=16)
    x_np = np.random.randn(2, 3, 16, 16).astype(np.float32)
    
    # Initialize NumPy layers
    conv_np = NumPyConv2D(in_channels=3, out_channels=4, kernel_size=3, stride=1, padding=1)
    pool_np = NumPyMaxPool2D(kernel_size=2, stride=2)
    
    # Initialize identical PyTorch layers
    conv_pt = nn.Conv2d(in_channels=3, out_channels=4, kernel_size=3, stride=1, padding=1)
    conv_pt.weight.data = torch.tensor(conv_np.W)
    conv_pt.bias.data = torch.tensor(conv_np.b)
    pool_pt = nn.MaxPool2d(kernel_size=2, stride=2)
    
    # PyTorch Forward
    x_pt = torch.tensor(x_np, requires_grad=True)
    out_pt_conv = conv_pt(x_pt)
    out_pt = pool_pt(out_pt_conv)
    loss_pt = out_pt.sum()
    loss_pt.backward()
    
    # NumPy Forward
    out_np_conv = conv_np.forward(x_np)
    out_np = pool_np.forward(out_np_conv)
    
    # NumPy Backward
    dout = np.ones_like(out_np)
    dpool = pool_np.backward(dout)
    dx_np, dW_np, db_np = conv_np.backward(dpool)
    
    # Verification checks
    forward_close = np.allclose(out_np, out_pt.detach().numpy(), atol=1e-5)
    grad_x_close = np.allclose(dx_np, x_pt.grad.numpy(), atol=1e-5)
    grad_w_close = np.allclose(dW_np, conv_pt.weight.grad.numpy(), atol=1e-5)
    grad_b_close = np.allclose(db_np, conv_pt.bias.grad.numpy(), atol=1e-5)
    
    print("=== NumPy vs PyTorch Autograd Verification ===")
    print(f"Forward Pass Match:           {forward_close}")
    print(f"Input Gradient (dX) Match:    {grad_x_close}")
    print(f"Weight Gradient (dW) Match:   {grad_w_close}")
    print(f"Bias Gradient (db) Match:     {grad_b_close}")

test_numpy_vs_pytorch()

In [ ]:
def setup_dataset(data_dir):
    # Search paths for Kaggle environments
    train_dir = os.path.join(data_dir, "train")
    val_dir = os.path.join(data_dir, "val")
    test_dir = os.path.join(data_dir, "test")

    # Collect paths from train and val subfolders
    all_train_val_paths = []
    all_train_val_labels = []

    for sub_dir in [train_dir, val_dir]:
        for cls_idx, cls_name in enumerate(["NORMAL", "PNEUMONIA"]):
            cls_path = os.path.join(sub_dir, cls_name)
            for img_path in glob.glob(os.path.join(cls_path, "*.jpeg")) + glob.glob(os.path.join(cls_path, "*.png")):
                all_train_val_paths.append(img_path)
                all_train_val_labels.append(cls_idx)

    # Perform stratified split (85% train, 15% validation)
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        all_train_val_paths,
        all_train_val_labels,
        test_size=0.15,
        stratify=all_train_val_labels,
        random_state=42
    )

    test_paths, test_labels = [], []
    for cls_idx, cls_name in enumerate(["NORMAL", "PNEUMONIA"]):
        cls_path = os.path.join(test_dir, cls_name)
        for img_path in glob.glob(os.path.join(cls_path, "*.jpeg")) + glob.glob(os.path.join(cls_path, "*.png")):
            test_paths.append(img_path)
            test_labels.append(cls_idx)

    print(f"Dataset summary:")
    print(f"  Train set size: {len(train_paths)} (Normal: {train_labels.count(0)}, Pneumonia: {train_labels.count(1)})")
    print(f"  Val set size:   {len(val_paths)} (Normal: {val_labels.count(0)}, Pneumonia: {val_labels.count(1)})")
    print(f"  Test set size:  {len(test_paths)} (Normal: {test_labels.count(0)}, Pneumonia: {test_labels.count(1)})")

    return (train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels)


class ChestXRayDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        image = Image.open(path).convert('RGB')
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
            
        return image, label


# Image preprocessing and augmentation transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [ ]:
class CustomCNN(nn.Module):
    """
    Custom 4-stage convolutional neural network.
    Conv2d -> BatchNorm -> ReLU -> MaxPool2d
    Followed by Adaptive Average Pooling and a single Logit linear output.
    """
    def __init__(self):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(128, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


def build_resnet18_transfer():
    """
    ResNet-18 fine-tuning setup with pre-trained ImageNet weights.
    Replaces default classification head with single binary logit unit.
    """
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_targets = [], []

    for images, targets in dataloader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        
        logits = model(images).squeeze(1)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.extend(probs)
        all_targets.extend(targets.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_targets, np.array(all_preds) >= 0.5)
    epoch_auc = roc_auc_score(all_targets, all_preds)
    return epoch_loss, epoch_acc, epoch_auc


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_logits, all_preds, all_targets = [], [], []

    for images, targets in dataloader:
        images, targets = images.to(device), targets.to(device)
        logits = model(images).squeeze(1)
        loss = criterion(logits, targets)

        running_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(logits).cpu().numpy()
        
        all_logits.extend(logits.cpu().numpy())
        all_preds.extend(probs)
        all_targets.extend(targets.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_targets, np.array(all_preds) >= 0.5)
    epoch_auc = roc_auc_score(all_targets, all_preds)
    return epoch_loss, epoch_acc, epoch_auc, np.array(all_logits), np.array(all_preds), np.array(all_targets)

In [ ]:
class TemperatureScaler(nn.Module):
    """
    Post-hoc calibration via single-parameter Temperature Scaling.
    Optimizes logit scaling parameter T on validation set using NLL.
    """
    def __init__(self):
        super(TemperatureScaler, self).__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

    def fit(self, val_logits, val_labels):
        val_logits_tensor = torch.tensor(val_logits, dtype=torch.float32)
        val_labels_tensor = torch.tensor(val_labels, dtype=torch.float32)
        
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.LBFGS([self.temperature], lr=0.01, max_iter=100)

        def eval_fn():
            optimizer.zero_grad()
            loss = criterion(self.forward(val_logits_tensor), val_labels_tensor)
            loss.backward()
            return loss

        optimizer.step(eval_fn)
        print(f"Optimal Temperature (T) parameter fitted: {self.temperature.item():.4f}")


def compute_calibration_metrics(probs, labels, n_bins=10):
    """
    Computes Expected Calibration Error (ECE) and Maximum Calibration Error (MCE).
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    mce = 0.0
    
    bin_accs, bin_confs, bin_counts = [], [], []

    for i in range(n_bins):
        bin_lower, bin_upper = bin_boundaries[i], bin_boundaries[i+1]
        in_bin = (probs > bin_lower) & (probs <= bin_upper)
        prop_in_bin = np.mean(in_bin)
        
        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(labels[in_bin])
            avg_confidence_in_bin = np.mean(probs[in_bin])
            abs_diff = np.abs(accuracy_in_bin - avg_confidence_in_bin)
            
            ece += abs_diff * prop_in_bin
            mce = max(mce, abs_diff)
            
            bin_accs.append(accuracy_in_bin)
            bin_confs.append(avg_confidence_in_bin)
            bin_counts.append(np.sum(in_bin))
        else:
            bin_accs.append(0.0)
            bin_confs.append(0.0)
            bin_counts.append(0)

    return ece, mce, bin_accs, bin_confs, bin_counts, bin_boundaries


def plot_reliability_diagrams(uncal_probs, cal_probs, labels, model_name="Model", n_bins=10):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    for ax, p, title in zip(axes, [uncal_probs, cal_probs], ["Uncalibrated", "Temperature Scaled"]):
        ece, mce, bin_accs, bin_confs, bin_counts, bin_boundaries = compute_calibration_metrics(p, labels, n_bins)
        
        bin_centers = (bin_boundaries[:-1] + bin_boundaries[1:]) / 2.0
        ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
        ax.bar(bin_centers, bin_accs, width=1.0/n_bins, alpha=0.6, color="royalblue", edgecolor="black", label="Outputs")
        ax.step(bin_centers, bin_confs, where="mid", color="crimson", linewidth=2, label="Confidence")
        
        ax.set_title(f"{model_name} ({title})\nECE: {ece*100:.2f}% | MCE: {mce*100:.2f}%")
        ax.set_xlabel("Predicted Probability")
        ax.set_ylabel("Empirical Accuracy")
        ax.legend(loc="upper left")
        ax.grid(True, linestyle=":", alpha=0.6)

    plt.tight_layout()
    plt.show()

In [ ]:
class GradCAM:
    """
    Computes Gradient-weighted Class Activation Mapping (Grad-CAM)
    visual heatmaps to identify spatial decision region focus.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, input_image):
        self.model.eval()
        output = self.model(input_image)
        self.model.zero_grad()
        output.backward()

        gradients = self.gradients.cpu().data.numpy()[0]
        activations = self.activations.cpu().data.numpy()[0]

        weights = np.mean(gradients, axis=(1, 2))
        cam = np.zeros(activations.shape[1:], dtype=np.float32)

        for i, w in enumerate(weights):
            cam += w * activations[i]

        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (224, 224)) if 'cv2' in globals() else cam
        cam = cam - np.min(cam)
        cam = cam / (np.max(cam) + 1e-8)
        return cam

In [ ]:
def compute_extended_metrics(probs, targets, threshold=0.5):
    """
    Turns raw probabilities into a threshold decision, then reports
    per-class precision/recall/F1 and the confusion matrix.
    This is what actually tells you the clinically relevant thing:
    how many real Pneumonia cases does the model MISS (false negatives)?
    """
    preds = (probs >= threshold).astype(int)

    # average=None -> return per-class arrays instead of one averaged number
    # labels=[0,1] -> force order [Normal, Pneumonia] regardless of what's present
    precision, recall, f1, support = precision_recall_fscore_support(
        targets, preds, average=None, labels=[0, 1], zero_division=0
    )
    cm = confusion_matrix(targets, preds, labels=[0, 1])
    return precision, recall, f1, support, cm


def print_extended_metrics(model_name, precision, recall, f1, support, cm):
    print(f"\n--- Per-Class Metrics: {model_name} (threshold=0.5, calibrated probs) ---")
    print(f"{'Class':<14}{'Precision':>10}{'Recall':>10}{'F1':>10}{'Support':>10}")
    print(f"{'Normal(0)':<14}{precision[0]:>10.3f}{recall[0]:>10.3f}{f1[0]:>10.3f}{support[0]:>10}")
    print(f"{'Pneumonia(1)':<14}{precision[1]:>10.3f}{recall[1]:>10.3f}{f1[1]:>10.3f}{support[1]:>10}")

    print(f"\nConfusion Matrix (rows=true label, cols=predicted) [Normal, Pneumonia]:")
    print(cm)

    # cm[1,0] = true Pneumonia (row 1), predicted Normal (col 0) -> a MISSED disease case
    fn = cm[1, 0]
    fp = cm[0, 1]
    print(f"False Negatives (missed Pneumonia): {fn}/{support[1]} ({fn/support[1]*100:.1f}%)")
    print(f"False Positives (Normal flagged as Pneumonia): {fp}/{support[0]} ({fp/support[0]*100:.1f}%)")

In [ ]:

# ============================================================
# Grad-CAM visualization 
# ============================================================


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

def disable_inplace_relu(model):
    """
    register_full_backward_hook wraps whatever tensor it's hooking into a
    special autograd node so it can capture the gradient passing through.
    An in-place ReLU (inplace=True) then immediately overwrites that same
    tensor's memory to produce its output -- which autograd refuses to allow
    on a hook-wrapped tensor, hence the "view is being modified inplace" error.
    Switching to inplace=False just makes ReLU allocate a fresh output tensor
    instead of overwriting its input. It computes the exact same function --
    same weights, same output values, marginally more memory -- so this is
    completely safe to do on a model that's already finished training.
    We walk every submodule recursively so this works regardless of whether
    the ReLU we need is a direct attribute (CustomCNN) or buried inside a
    container we're hooking as a whole (ResNet's BasicBlock).
    """
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            module.inplace = False
    return model

def unnormalize_for_display(tensor_img):
    """Undo the ImageNet normalization so matplotlib shows a real-looking image."""
    img = tensor_img.cpu().numpy().transpose(1, 2, 0)   # (C,H,W) -> (H,W,C) for matplotlib
    img = img * IMAGENET_STD + IMAGENET_MEAN            # reverse the Normalize() transform
    return np.clip(img, 0, 1)                           # clip floating rounding error back to valid [0,1]

def get_target_layer(model, model_name):
    """Pick the last conv layer to hook — this is the conventional Grad-CAM choice
    (last conv block = most spatially-resolved layer that still has semantic/class info)."""
    if model_name == "Custom_CNN":
        return model.features[14]     # last ReLU, i.e. output of the 4th conv block
    else:
        return model.layer4[-1]       # ResNet18's final residual block

def show_gradcam_examples(model, model_name, test_dataset, test_labels, n_per_class=2):
    disable_inplace_relu(model)
    model.eval()
    target_layer = get_target_layer(model, model_name)
    cam_extractor = GradCAM(model, target_layer)

    # grab a few NORMAL (label 0) and PNEUMONIA (label 1) example indices
    normal_idx = [i for i, l in enumerate(test_labels) if l == 0][:n_per_class]
    pneu_idx   = [i for i, l in enumerate(test_labels) if l == 1][:n_per_class]
    sample_idx = normal_idx + pneu_idx

    fig, axes = plt.subplots(2, len(sample_idx), figsize=(4 * len(sample_idx), 8))

    for col, idx in enumerate(sample_idx):
        img_tensor, label = test_dataset[idx]
        input_batch = img_tensor.unsqueeze(0).to(device)   # add batch dim: (C,H,W) -> (1,C,H,W)

        cam = cam_extractor.generate(input_batch)           # (224,224) heatmap, values in [0,1]
        display_img = unnormalize_for_display(img_tensor)

        # top row: raw image
        axes[0, col].imshow(display_img)
        axes[0, col].set_title(f"True: {'Pneumonia' if label==1 else 'Normal'}")
        axes[0, col].axis('off')

        # bottom row: image with heatmap overlaid
        axes[1, col].imshow(display_img)
        axes[1, col].imshow(cam, cmap='jet', alpha=0.45)     # alpha blends heatmap over the X-ray
        axes[1, col].set_title("Grad-CAM")
        axes[1, col].axis('off')

    fig.suptitle(f"Grad-CAM — {model_name}", fontsize=14)
    plt.tight_layout()
    plt.show()



In [ ]:
# --- Locate the dataset (robust to Kaggle's "+ Add Data" mount vs. a local/Colab copy) ---
def find_chest_xray_root(candidates):
    """Return the first candidate directory that actually contains a train/ subfolder."""
    for c in candidates:
        if os.path.isdir(c) and os.path.isdir(os.path.join(c, "train")):
            return c
    return None

candidate_paths = [
    "/kaggle/input/chest-xray-pneumonia/chest_xray",
    "/kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray",  # some dataset versions nest an extra level
    "./chest_xray",
]

data_dir = find_chest_xray_root(candidate_paths)

if data_dir is None:
    # Only useful outside Kaggle's own hosted notebooks (e.g. Colab), where internet
    # access is available. On Kaggle itself, internet is off by default for notebook
    # execution, and it's unnecessary anyway -- see the error message below.
    print("Dataset not found locally. Attempting to download via Kaggle API "
          "(requires internet access + a configured kaggle.json -- typically "
          "only relevant on Colab, not Kaggle's own hosted notebooks)...")
    exit_code = os.system("kaggle datasets download -d paultimothymooney/chest-xray-pneumonia")
    if exit_code == 0 and os.path.exists("chest-xray-pneumonia.zip"):
        os.system("unzip -q -o chest-xray-pneumonia.zip -d ./chest_xray_temp")
        data_dir = find_chest_xray_root(["./chest_xray_temp/chest_xray", "./chest_xray_temp"])

if data_dir is None:
    raise FileNotFoundError(
        "Could not locate the chest X-ray dataset.\n\n"
        "If you're running this on Kaggle: click '+ Add Data' in the right-hand "
        "sidebar of the notebook editor, search for "
        "'Chest X-Ray Images (Pneumonia)' (paultimothymooney/chest-xray-pneumonia), "
        "and click Add. Kaggle will mount it automatically at "
        "/kaggle/input/chest-xray-pneumonia/ -- no download code needed, just re-run "
        "this cell afterward.\n\n"
        "If you're on Colab instead: upload your kaggle.json API token "
        "(from kaggle.com/settings) and run:\n"
        "  !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json\n"
        "before this cell."
    )

print(f"Using dataset directory: {data_dir}")
print("Top-level contents:", os.listdir(data_dir))

# Data setup
(train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels) = setup_dataset(data_dir)

train_dataset = ChestXRayDataset(train_paths, train_labels, transform=data_transforms['train'])
val_dataset = ChestXRayDataset(val_paths, val_labels, transform=data_transforms['val'])
test_dataset = ChestXRayDataset(test_paths, test_labels, transform=data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Compute pos_weight for BCEWithLogitsLoss to address class imbalance
num_negatives = train_labels.count(0)
num_positives = train_labels.count(1)
pos_weight_val = torch.tensor([num_negatives / num_positives], dtype=torch.float32).to(device)
print(f"Class imbalance pos_weight computed for loss function: {pos_weight_val.item():.4f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_val)

# Instantiate models
models_dict = {
    "Custom_CNN": (CustomCNN().to(device), 1e-3),
    "ResNet18_Pretrained": (build_resnet18_transfer().to(device), 1e-4)
}

results_summary = []

for model_name, (model, lr) in models_dict.items():
    print(f"\n==========================================")
    print(f"       Training Model: {model_name}")
    print(f"==========================================")
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    best_val_loss = float('inf')
    best_weights = copy.deepcopy(model.state_dict())
    patience, patience_counter = 5, 0

    for epoch in range(1, 26):
        t_loss, t_acc, t_auc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        v_loss, v_acc, v_auc, _, _, _ = evaluate(model, val_loader, criterion, device)
        
        print(f"Epoch {epoch:02d} | Train Loss: {t_loss:.4f} Acc: {t_acc*100:.2f}% AUC: {t_auc:.4f} | "
              f"Val Loss: {v_loss:.4f} Acc: {v_acc*100:.2f}% AUC: {v_auc:.4f}")

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch}.")
                break
        scheduler.step(v_loss)
    model.load_state_dict(best_weights)
    
    show_gradcam_examples(model, model_name, test_dataset, test_labels)
    
    # Collect Validation Logits for Temperature Calibration Fitting
    _, _, _, val_logits, _, val_targets = evaluate(model, val_loader, criterion, device)
    
    # Fit Temperature Scaler on Validation Set
    scaler = TemperatureScaler()
    scaler.fit(val_logits, val_targets)
    
    # Evaluate on Unseen Held-Out Test Set
    _, test_acc, test_auc, test_logits, test_probs_uncal, test_targets = evaluate(model, test_loader, criterion, device)
    
    # Temperature Scaled Predictions
    test_logits_scaled = scaler(torch.tensor(test_logits, dtype=torch.float32)).detach().numpy()
    test_probs_cal = 1.0 / (1.0 + np.exp(-test_logits_scaled))

    # Calculate Calibration Errors
    ece_uncal, mce_uncal, _, _, _, _ = compute_calibration_metrics(test_probs_uncal, test_targets)
    ece_cal, mce_cal, _, _, _, _ = compute_calibration_metrics(test_probs_cal, test_targets)

    # Minority Class Calibration Check (Normal Class = 0)
    normal_idx = (test_targets == 0)
    ece_minority_uncal, _, _, _, _, _ = compute_calibration_metrics(test_probs_uncal[normal_idx], test_targets[normal_idx])
    ece_minority_cal, _, _, _, _, _ = compute_calibration_metrics(test_probs_cal[normal_idx], test_targets[normal_idx])

    precision, recall, f1, support, cm = compute_extended_metrics(test_probs_cal, test_targets)
    print_extended_metrics(model_name, precision, recall, f1, support, cm)

    print(f"\n--- Held-Out Test Set Evaluation: {model_name} ---")
    print(f"Test Accuracy:           {test_acc*100:.2f}%")
    print(f"Test ROC-AUC:            {test_auc:.4f}")
    print(f"Uncalibrated ECE:        {ece_uncal*100:.2f}% | MCE: {mce_uncal*100:.2f}%")
    print(f"Calibrated ECE (T={scaler.temperature.item():.2f}): {ece_cal*100:.2f}% | MCE: {mce_cal*100:.2f}%")
    print(f"Minority Class (Normal) ECE Before: {ece_minority_uncal*100:.2f}% | After: {ece_minority_cal*100:.2f}%")

    # Plot Reliability Diagrams
    plot_reliability_diagrams(test_probs_uncal, test_probs_cal, test_targets, model_name=model_name)

    results_summary.append({
        "Model": model_name,
        "Accuracy": f"{test_acc*100:.2f}%",
        "ROC-AUC": f"{test_auc:.4f}",
        "Recall (Pneumonia)": f"{recall[1]*100:.2f}%",  
        "Precision (Pneumonia)": f"{precision[1]*100:.2f}%", 
        "Optimal T": f"{scaler.temperature.item():.3f}",
        "ECE Before": f"{ece_uncal*100:.2f}%",
        "ECE After": f"{ece_cal*100:.2f}%",
        "MCE Before": f"{mce_uncal*100:.2f}%",
        "MCE After": f"{mce_cal*100:.2f}%",
        "Minority ECE After": f"{ece_minority_cal*100:.2f}%"
    })

# Render final summary comparison table
df_results = pd.DataFrame(results_summary)
print("\n================ FINAL COMPARATIVE CALIBRATION RESULTS ================")
print(df_results.to_markdown(index=False))
print(f"""
Note on calibration (auto-generated from THIS run's actual numbers -- do not
hardcode approximate figures here again, print df_results directly instead):
{df_results.to_markdown(index=False)}

Temperature scaling moved overall ECE only marginally for both models this run
(see ECE Before/After above), and did not meaningfully fix minority-class
(Normal) calibration in either model. This matches the known limitation of
vanilla temperature scaling under class imbalance: one global scalar can only
rescale the overall sharpness of a model's confidence, not correct a
systematic bias that differs between classes. Fixing minority-class
calibration specifically would need per-class (vector) temperature scaling
instead of one shared T -- a natural next step, not attempted here.

Reproducibility note: re-running this notebook has produced noticeably
different calibration numbers between runs (see README.md's Reproducibility
Note for a side-by-side of two actual runs) -- report whichever run's numbers
you cite alongside that caveat, don't treat a single run as definitive.
""")